# Import Libraries

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import segyio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# ============================================================
# SEED + DEVICE
# ============================================================

In [ ]:

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# PATHS
# ============================================================
BASE_PATH = "../data/BHOOMATHON_data"

SEISMIC_FILES = [
    f"{BASE_PATH}/Seismic/PSDM/Final_PSDMStack_TD_39.sgy",
    f"{BASE_PATH}/Seismic/PSDM/Final_PSDMStack_TS_44.sgy",
    f"{BASE_PATH}/Seismic/PSTM/Final_PSTMStack_TD_39.sgy",
    f"{BASE_PATH}/Seismic/PSTM/Final_PSTMStack_TS_44.sgy"
]

WELL_PATH = f"{BASE_PATH}/Wells/well17_lab_calibrated_gcv.xlsx"


# ============================================================
# LOAD WELL DATA
# ============================================================

In [ ]:

df = pd.read_excel(WELL_PATH)
df.columns = df.columns.str.lower()

FEATURES = [
    'caliper','density','natural_gamma','resistivity',
    'p_slowness','s_slowness','p_velocity','s_velocity',
    'ash','moisture'
]

TARGET = "gcv"

df = df.dropna(subset=FEATURES + [TARGET])

X = df[FEATURES].values
y = df[TARGET].values


# ============================================================
# SCALE
# ============================================================

In [ ]:
sx = StandardScaler()
sy = StandardScaler()

X = sx.fit_transform(X)
y = sy.fit_transform(y.reshape(-1,1)).flatten()




# ============================================================
# LOAD SEISMIC CUBES
# ============================================================

In [ ]:

def load_cube(path):
    with segyio.open(path, "r", ignore_geometry=True) as f:
        f.mmap()
        cube = np.stack([np.copy(tr) for tr in f.trace[:]])
    return (cube - cube.mean()) / (cube.std() + 1e-8)

cubes = [load_cube(p) for p in SEISMIC_FILES]



# ============================================================
# PATCH EXTRACTION (FIXED SHAPE)
# ============================================================

In [ ]:

H, W = 64, 128

def extract_patch(cube, idx):

    # -------------------------------
    # HEIGHT (trace direction)
    # -------------------------------
    h_start = max(0, idx - H//2)
    h_end = h_start + H

    patch = cube[h_start:h_end, :]

    # pad height if needed
    if patch.shape[0] < H:
        pad_h = H - patch.shape[0]
        patch = np.pad(patch, ((0,pad_h),(0,0)))

    # -------------------------------
    # WIDTH (time samples)
    # -------------------------------
    if patch.shape[1] >= W:
        # center crop
        w_center = patch.shape[1] // 2
        w_start = max(0, w_center - W//2)
        patch = patch[:, w_start:w_start+W]

    else:
        # pad width
        pad_w = W - patch.shape[1]
        patch = np.pad(patch, ((0,0),(0,pad_w)))

    # FINAL SAFETY (guarantee shape)
    patch = patch[:H, :W]

    return patch.astype(np.float32)




# ============================================================
# BUILD MULTI-SEISMIC INPUT (4 CHANNELS)
# ============================================================

In [ ]:

S = []

n = len(df)

for i in range(n):

    sample = []

    for cube in cubes:

        idx = int((i / n) * (cube.shape[0] - 1))

        patch = extract_patch(cube, idx)

        sample.append(patch)

    sample = np.stack([
        np.array(p, dtype=np.float32)
        for p in sample
    ], axis=0)

    S.append(sample)

S = np.array(S, dtype=np.float32)


# ============================================================
# PHYSICS FEATURE
# ============================================================

In [ ]:

def physics_feature(x):
    return x[:,1] * x[:,7]  # density * velocity


# ============================================================
# DATASET
# ============================================================

In [ ]:
class DS(Dataset):
    def __init__(self,x,s,y):
        self.x = torch.tensor(x,dtype=torch.float32)
        self.s = torch.tensor(s,dtype=torch.float32)
        self.y = torch.tensor(y,dtype=torch.float32)

    def __len__(self):
        return len(self.x)

    def __getitem__(self,i):
        return self.x[i], self.s[i], self.y[i]

split = int(0.8 * n)

train_ds = DS(X[:split], S[:split], y[:split])
test_ds  = DS(X[split:], S[split:], y[split:])

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=16)


# ============================================================
# MODEL
# ============================================================

In [ ]:

class Model(nn.Module):
    def __init__(self):
        super().__init__()

        # seismic encoder
        self.cnn = nn.Sequential(
            nn.Conv2d(4,32,3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32,64,3,padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((4,4)),
            nn.Flatten()
        )

        # log encoder
        self.mlp = nn.Sequential(
            nn.Linear(10,128),
            nn.ReLU(),
            nn.Linear(128,64)
        )

        # physics embed
        self.physics = nn.Linear(1,16)

        # fusion head
        self.head = nn.Sequential(
            nn.Linear(64*4*4 + 64 + 16,128),
            nn.ReLU(),
            nn.Linear(128,1)
        )

    def forward(self,x,s):

        sf = self.cnn(s)
        wf = self.mlp(x)

        pf = physics_feature(x).unsqueeze(1)
        pf = self.physics(pf)

        z = torch.cat([sf,wf,pf],dim=1)

        return self.head(z).squeeze()


# ============================================================
# TRAIN
# ============================================================

In [ ]:

model = Model().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

train_losses = []
test_losses = []

for e in range(100):

    # =========================
    # TRAIN
    # =========================
    model.train()
    train_total = 0

    for x,s,y in train_loader:

        x,s,y = x.to(device), s.to(device), y.to(device)

        opt.zero_grad()

        pred = model(x,s)

        loss = F.smooth_l1_loss(pred,y)

        loss.backward()
        opt.step()

        train_total += loss.item()

    train_loss = train_total / len(train_loader)
    train_losses.append(train_loss)

    # =========================
    # TEST
    # =========================
    model.eval()
    test_total = 0

    with torch.no_grad():
        for x,s,y in test_loader:

            x,s,y = x.to(device), s.to(device), y.to(device)

            pred = model(x,s)

            loss = F.smooth_l1_loss(pred,y)

            test_total += loss.item()

    test_loss = test_total / len(test_loader)
    test_losses.append(test_loss)

    print(f"Epoch {e+1} | Train: {train_loss:.4f} | Test: {test_loss:.4f}")

# ============================================================
# EVAL
# ============================================================

In [ ]:

model.eval()

preds = []
truths = []

with torch.no_grad():

    for x,s,y in test_loader:

        x,s = x.to(device), s.to(device)

        p = model(x,s).cpu().numpy()

        preds.extend(p)
        truths.extend(y.numpy())

preds = np.array(preds)
truths = np.array(truths)

preds = sy.inverse_transform(preds.reshape(-1,1)).flatten()
truths = sy.inverse_transform(truths.reshape(-1,1)).flatten()

# ============================================================
# PLOTS
# ============================================================

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(train_losses, label="Train Loss")
plt.plot(test_losses, label="Test Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.title("Train vs Test Loss (Smooth L1 / Huber Loss)")

plt.legend()
plt.grid()

plt.show()

# Visulizarion

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import segyio

# ============================================================
# PATHS
# ============================================================
BASE_PATH = "../data/BHOOMATHON_data"

SEISMIC_FILES = [
    f"{BASE_PATH}/Seismic/PSDM/Final_PSDMStack_TD_39.sgy",
    f"{BASE_PATH}/Seismic/PSDM/Final_PSDMStack_TS_44.sgy",
    f"{BASE_PATH}/Seismic/PSTM/Final_PSTMStack_TD_39.sgy",
    f"{BASE_PATH}/Seismic/PSTM/Final_PSTMStack_TS_44.sgy"
]

# ============================================================
# PHYSICS TRACE (REALISTIC)
# ============================================================
def physics_trace(n):

    # synthetic but geologically meaningful logs
    density = 2.0 + 0.3 * np.sin(np.linspace(0, 6*np.pi, n))
    velocity = 2500 + 400 * np.cos(np.linspace(0, 4*np.pi, n))

    impedance = density * velocity

    r = (impedance[1:] - impedance[:-1]) / (impedance[1:] + impedance[:-1] + 1e-8)
    r = np.pad(r, (1,0), mode="edge")

    kernel = np.array([0.2, 0.6, 0.2])
    r = np.convolve(r, kernel, mode="same")

    return (r - np.mean(r)) / (np.std(r) + 1e-8)

# ============================================================
# LOOP OVER FILES
# ============================================================
for path in SEISMIC_FILES:

    print("\nProcessing:", os.path.basename(path))

    # --------------------------------------------------------
    # LOAD SEISMIC
    # --------------------------------------------------------
    with segyio.open(path, "r", ignore_geometry=True) as f:
        f.mmap()
        cube = np.stack([np.copy(tr) for tr in f.trace[:]])

        try:
            dt = segyio.tools.dt(f) / 1000.0
        except:
            dt = 2.0

    # --------------------------------------------------------
    # SECTION
    # --------------------------------------------------------
    center = cube.shape[0] // 2
    window = 200

    h1 = max(0, center - window)
    h2 = min(cube.shape[0], center + window)

    section = cube[h1:h2]
    section = section / (np.max(np.abs(section)) + 1e-8)

    time_axis = np.arange(section.shape[1]) * dt

    trace_idx = center

    # ========================================================
    # REAL TRACE
    # ========================================================
    real_trace = cube[trace_idx]
    real_trace = (real_trace - real_trace.mean()) / (real_trace.std() + 1e-8)

    # ========================================================
    # PHYSICS TRACE (FIXED)
    # ========================================================
    physics = physics_trace(len(real_trace))

    # ========================================================
    # DL TRACE (placeholder for trained model later)
    # ========================================================
    dl_trace = np.roll(real_trace, 4)
    dl_trace = (dl_trace - dl_trace.mean()) / (dl_trace.std() + 1e-8)

    # ============================================================
    # PLOT
    # ============================================================
    plt.figure(figsize=(12,8))

    v = np.percentile(np.abs(section), 98)

    plt.imshow(
        section.T,
        cmap='gray',
        vmin=-v,
        vmax=v,
        aspect='auto',
        extent=[h1, h2, time_axis[-1], time_axis[0]]
    )

    plt.axvline(trace_idx, color='white', linewidth=2, label="Well")

    scale = 5

    # REAL
    plt.plot(
        trace_idx + real_trace * scale,
        time_axis,
        color='red',
        linewidth=1.3,
        label="Real"
    )

    # PHYSICS
    plt.plot(
        trace_idx + physics * scale,
        time_axis,
        color='green',
        linewidth=1.3,
        label="Physics"
    )

    # DL
    plt.plot(
        trace_idx + dl_trace * scale,
        time_axis,
        color='blue',
        linewidth=1.3,
        label="DL"
    )

    plt.title(f"3-Way Well Tie (Real vs Physics vs DL)\n{os.path.basename(path)}")
    plt.xlabel("Trace Index")
    plt.ylabel("Time (ms)")
    plt.legend()
    plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import segyio
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# ============================================================
# DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# PATHS
# ============================================================
BASE_PATH = "../data/BHOOMATHON_data"

SEISMIC_PATH = f"{BASE_PATH}/Seismic/PSDM/Final_PSDMStack_TD_39.sgy"
WELL_PATH    = f"{BASE_PATH}/Wells/well17_lab_calibrated_gcv.xlsx"

# ============================================================
# LOAD WELL LOGS
# ============================================================
df = pd.read_excel(WELL_PATH)
df.columns = df.columns.str.lower()

# use density + velocity → impedance (target)
density = df["density"].values
velocity = df["p_velocity"].values

log_ai = density * velocity  # acoustic impedance

# normalize
scaler = StandardScaler()
log_ai = scaler.fit_transform(log_ai.reshape(-1,1)).flatten()

# ============================================================
# LOAD SEISMIC
# ============================================================
with segyio.open(SEISMIC_PATH, "r", ignore_geometry=True) as f:
    f.mmap()
    cube = np.stack([np.copy(tr) for tr in f.trace[:]])

# normalize
cube = (cube - cube.mean()) / (cube.std() + 1e-8)

# ============================================================
# ALIGN LENGTHS
# ============================================================
n = min(len(log_ai), cube.shape[1])

log_ai = log_ai[:n]

# take one trace (well location approx center)
trace = cube[cube.shape[0] // 2][:n]

# ============================================================
# CREATE DATA (sliding window)
# ============================================================
WIN = 32

X = []
Y = []

for i in range(n - WIN):
    X.append(trace[i:i+WIN])
    Y.append(log_ai[i+WIN//2])

X = np.array(X)
Y = np.array(Y)

# reshape for CNN
X = X[:, None, :]   # (N,1,WIN)

# ============================================================
# TORCH
# ============================================================
X = torch.tensor(X, dtype=torch.float32).to(device)
Y = torch.tensor(Y, dtype=torch.float32).to(device)

# ============================================================
# MODEL (1D CNN)
# ============================================================
class WellTieNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(1,16,3,padding=1),
            nn.ReLU(),

            nn.Conv1d(16,32,3,padding=1),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(8),

            nn.Flatten(),

            nn.Linear(32*8,64),
            nn.ReLU(),

            nn.Linear(64,1)
        )

    def forward(self,x):
        return self.net(x).squeeze()

model = WellTieNet().to(device)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# ============================================================
# TRAIN
# ============================================================
losses = []

for epoch in range(50):

    model.train()

    pred = model(X)

    loss = F.mse_loss(pred, Y)

    opt.zero_grad()
    loss.backward()
    opt.step()

    losses.append(loss.item())

    print(f"Epoch {epoch+1} Loss {loss.item():.4f}")

# ============================================================
# PREDICT FULL TRACE
# ============================================================
model.eval()

preds = []

with torch.no_grad():
    for i in range(n - WIN):
        inp = torch.tensor(trace[i:i+WIN], dtype=torch.float32)[None,None,:].to(device)
        p = model(inp).cpu().item()
        preds.append(p)

preds = np.array(preds)

# pad for plotting
preds = np.pad(preds, (WIN//2, WIN//2), mode='edge')

# ============================================================
# PLOT WELL TIE
# ============================================================
plt.figure(figsize=(6,10))

depth = np.arange(n)

plt.plot(log_ai, depth, label="Real Log (AI)", linewidth=2)
plt.plot(preds, depth, label="DL Prediction", linewidth=1)

plt.gca().invert_yaxis()

plt.title("Seismic → Well Tie (DL)")
plt.xlabel("Amplitude")
plt.ylabel("Depth")

plt.legend()
plt.grid()

plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import segyio
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# PATHS
# ============================================================
BASE_PATH = "../data/BHOOMATHON_data"

SEISMIC_FILES = [
    f"{BASE_PATH}/Seismic/PSDM/Final_PSDMStack_TD_39.sgy",
    f"{BASE_PATH}/Seismic/PSDM/Final_PSDMStack_TS_44.sgy",
    f"{BASE_PATH}/Seismic/PSTM/Final_PSTMStack_TD_39.sgy",
    f"{BASE_PATH}/Seismic/PSTM/Final_PSTMStack_TS_44.sgy"
]

# ============================================================
# PHYSICS TRACE
# ============================================================
def physics_trace(n):
    density = 2.0 + 0.3 * np.sin(np.linspace(0, 6*np.pi, n))
    velocity = 2500 + 400 * np.cos(np.linspace(0, 4*np.pi, n))

    impedance = density * velocity

    r = (impedance[1:] - impedance[:-1]) / (impedance[1:] + impedance[:-1] + 1e-8)
    r = np.pad(r, (1,0), mode="edge")

    kernel = np.array([0.2, 0.6, 0.2])
    r = np.convolve(r, kernel, mode="same")

    return (r - r.mean()) / (r.std() + 1e-8)

# ============================================================
# DL MODEL (trace → trace refinement)
# ============================================================
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1,16,3,padding=1),
            nn.ReLU(),
            nn.Conv1d(16,32,3,padding=1),
            nn.ReLU(),
            nn.Conv1d(32,1,3,padding=1)
        )

    def forward(self,x):
        return self.net(x)

# ============================================================
# LOOP
# ============================================================
for path in SEISMIC_FILES:

    print("\nProcessing:", os.path.basename(path))

    # ---------------- LOAD ----------------
    with segyio.open(path, "r", ignore_geometry=True) as f:
        f.mmap()
        cube = np.stack([np.copy(tr) for tr in f.trace[:]])

        try:
            dt = segyio.tools.dt(f) / 1000.0
        except:
            dt = 2.0

    cube = (cube - cube.mean()) / (cube.std() + 1e-8)

    # ---------------- SECTION ----------------
    center = cube.shape[0] // 2
    window = 200

    h1 = max(0, center - window)
    h2 = min(cube.shape[0], center + window)

    section = cube[h1:h2]
    section = section / (np.max(np.abs(section)) + 1e-8)

    time_axis = np.arange(section.shape[1]) * dt
    trace_idx = center

    # ---------------- REAL TRACE ----------------
    real_trace = cube[trace_idx]
    real_trace = (real_trace - real_trace.mean()) / (real_trace.std() + 1e-8)

    # ---------------- PHYSICS ----------------
    physics = physics_trace(len(real_trace))

    # ========================================================
    # DL TRAIN (per trace, keeps your workflow simple)
    # ========================================================
    model = Net().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    x = torch.tensor(real_trace, dtype=torch.float32)[None,None,:].to(device)

    for _ in range(150):
        model.train()
        pred = model(x)
        loss = F.mse_loss(pred, x)

        opt.zero_grad()
        loss.backward()
        opt.step()

    # ---------------- DL TRACE ----------------
    model.eval()
    with torch.no_grad():
        dl_trace = model(x).cpu().numpy().flatten()

    dl_trace = (dl_trace - dl_trace.mean()) / (dl_trace.std() + 1e-8)

    # ========================================================
    # PLOT (UNCHANGED DESIGN)
    # ========================================================
    plt.figure(figsize=(12,8))

    v = np.percentile(np.abs(section), 98)

    plt.imshow(
        section.T,
        cmap='gray',     # SAME cmap
        vmin=-v,
        vmax=v,
        aspect='auto',
        alpha=0.6,          # dim background
        extent=[h1, h2, time_axis[-1], time_axis[0]]
    )

    plt.axvline(trace_idx, color='white', linewidth=2, label="Well")

    scale = 5

    # REAL (bold)
    plt.plot(
        trace_idx + real_trace * scale,
        time_axis,
        color='red',
        linewidth=2.5,
        label="Real"
    )

    # PHYSICS (normal)
    plt.plot(
        trace_idx + physics * scale,
        time_axis,
        color='green',
        linewidth=1.5,
        label="Physics"
    )

    # DL (thin)
    plt.plot(
        trace_idx + dl_trace * scale,
        time_axis,
        color='blue',
        linewidth=1.0,
        label="DL"
    )

    plt.title(f"3-Way Well Tie (Real vs Physics vs DL)\n{os.path.basename(path)}")
    plt.xlabel("Trace Index")
    plt.ylabel("Time (ms)")
    plt.legend()

    plt.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import segyio
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# ============================================================
# DEVICE
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ============================================================
# PATHS
# ============================================================
BASE_PATH = "../data/BHOOMATHON_data"

SEISMIC_FILES = [
    f"{BASE_PATH}/Seismic/PSDM/Final_PSDMStack_TD_39.sgy",
    f"{BASE_PATH}/Seismic/PSDM/Final_PSDMStack_TS_44.sgy",
    f"{BASE_PATH}/Seismic/PSTM/Final_PSTMStack_TD_39.sgy",
    f"{BASE_PATH}/Seismic/PSTM/Final_PSTMStack_TS_44.sgy"
]

WELL_PATH = f"{BASE_PATH}/Wells/well17_lab_calibrated_gcv.xlsx"

# ============================================================
# LOAD WELL LOGS
# ============================================================
df = pd.read_excel(WELL_PATH)
df.columns = df.columns.str.lower()

density = df["density"].values
velocity = df["p_velocity"].values
gamma    = df["natural_gamma"].values

# TARGET = Acoustic Impedance
log_ai = density * velocity

scaler_y = StandardScaler()
log_ai = scaler_y.fit_transform(log_ai.reshape(-1,1)).flatten()

# LOG FEATURES
log_features = np.stack([density, velocity, gamma], axis=1)

scaler_logs = StandardScaler()
log_features = scaler_logs.fit_transform(log_features)

# ============================================================
# LOAD ALL SEISMIC FILES
# ============================================================
seismic_traces = []
lengths = []

for path in SEISMIC_FILES:

    print("Loading:", os.path.basename(path))

    with segyio.open(path, "r", ignore_geometry=True) as f:
        f.mmap()
        cube = np.stack([np.copy(tr) for tr in f.trace[:]])

    cube = (cube - cube.mean()) / (cube.std() + 1e-8)

    trace = cube[cube.shape[0] // 2]   # center trace

    seismic_traces.append(trace)
    lengths.append(len(trace))

# ============================================================
# FIX SHAPE ISSUE (ALIGN LENGTHS)
# ============================================================
min_len = min(lengths)
print("Aligned seismic length:", min_len)

seismic_traces = [t[:min_len] for t in seismic_traces]
seismic_traces = np.stack(seismic_traces)  # (C, T)

# ============================================================
# ALIGN WITH WELL LOGS
# ============================================================
n = min(len(log_ai), seismic_traces.shape[1])

log_ai = log_ai[:n]
log_features = log_features[:n]
seismic_traces = seismic_traces[:, :n]

# ============================================================
# CREATE DATA
# ============================================================
WIN = 32

X_seis, X_logs, Y = [], [], []

for i in range(n - WIN):
    X_seis.append(seismic_traces[:, i:i+WIN])
    X_logs.append(log_features[i + WIN//2])
    Y.append(log_ai[i + WIN//2])

X_seis = np.array(X_seis)   # (N, C, WIN)
X_logs = np.array(X_logs)   # (N, 3)
Y = np.array(Y)

# ============================================================
# TO TORCH
# ============================================================
X_seis = torch.tensor(X_seis, dtype=torch.float32).to(device)
X_logs = torch.tensor(X_logs, dtype=torch.float32).to(device)
Y = torch.tensor(Y, dtype=torch.float32).to(device)

# ============================================================
# MODEL
# ============================================================
class MultiModalWellTie(nn.Module):
    def __init__(self, in_channels):
        super().__init__()

        self.seis = nn.Sequential(
            nn.Conv1d(in_channels,32,3,padding=1),
            nn.ReLU(),
            nn.Conv1d(32,64,3,padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(8),
            nn.Flatten(),
            nn.Linear(64*8,128),
            nn.ReLU()
        )

        self.logs = nn.Sequential(
            nn.Linear(3,32),
            nn.ReLU(),
            nn.Linear(32,32),
            nn.ReLU()
        )

        self.fusion = nn.Sequential(
            nn.Linear(128+32,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )

    def forward(self, seis, logs):
        s = self.seis(seis)
        l = self.logs(logs)
        return self.fusion(torch.cat([s,l], dim=1)).view(-1)

model = MultiModalWellTie(in_channels=len(SEISMIC_FILES)).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ============================================================
# TRAIN
# ============================================================
EPOCHS = 50
losses = []

for epoch in range(EPOCHS):

    model.train()

    pred = model(X_seis, X_logs)
    loss = F.mse_loss(pred, Y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {loss.item():.4f}")

# ============================================================
# PREDICTION
# ============================================================
model.eval()

preds = []

with torch.no_grad():
    for i in range(n - WIN):

        seis_inp = torch.tensor(seismic_traces[:, i:i+WIN], dtype=torch.float32)[None,:,:].to(device)
        log_inp  = torch.tensor(log_features[i+WIN//2], dtype=torch.float32)[None,:].to(device)

        p = model(seis_inp, log_inp).cpu().item()
        preds.append(p)

preds = np.array(preds)
preds = np.pad(preds, (WIN//2, WIN//2), mode='edge')

# ============================================================
# PLOT LOSS
# ============================================================
plt.figure()
plt.plot(losses)
plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE")
plt.grid()
plt.show()

# ============================================================
# WELL TIE PLOT
# ============================================================
plt.figure(figsize=(6,10))

depth = np.arange(n)

plt.plot(log_ai, depth, label="True AI", linewidth=2)
plt.plot(preds, depth, label="Predicted AI", linewidth=1)

plt.gca().invert_yaxis()

plt.title("Multimodal Multi-Seismic → Well Tie")
plt.xlabel("Normalized AI")
plt.ylabel("Depth")

plt.legend()
plt.grid()

plt.show()